# Pretest analysis

Analysis code for Bui Ngoc Minh Quach’s 2026 master’s thesis at the Technical University of Munich. See the repository README for setup, data access and citation.

# 0. Setup and Environment

This chapter imports the required libraries, defines the project folders, and loads the raw Qualtrics pretest dataset without modification.

## 0.1 Required packages

Install the Python dependencies from the repository’s `requirements.txt` before starting JupyterLab.

## 0.2 Import Libraries

All package imports used in the notebook are consolidated below.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print("✓ Libraries successfully imported.")

## 0.3 Project Structure

Relative project folders are used so that the notebook can be run from any location.

In [ ]:
# Locate the repository from its root or the notebooks folder.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise RuntimeError("Start JupyterLab from the repository root.")

DATA_DIR = PROJECT_DIR / "data" / "pretest"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "pretest"
TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 0.4 Load the Raw Dataset

The raw Qualtrics export is loaded from `data/Pretest_Dataset.csv`.

In [ ]:
DATA_FILE = DATA_DIR / "Pretest_Dataset.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        "This notebook requires an authorized local copy of "
        "data/pretest/Pretest_Dataset.csv. See the repository README."
    )

df_raw = pd.read_csv(DATA_FILE)
print(f"Loaded {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")


## 0.5 Chapter Summary

The analysis environment and project folders are configured, and the raw pretest dataset is loaded.

# 1. Data Preparation

This chapter removes the two Qualtrics metadata rows and applies the pretest screening criteria. The analytical sample includes consenting adult respondents who completed the instructed-response attention check.

## 1.1 Remove Qualtrics Metadata

The first two rows contain Qualtrics question labels and import identifiers rather than participant responses.

In [ ]:
df = df_raw.iloc[2:].copy().reset_index(drop=True)

print(f"Raw export rows       : {len(df_raw)}")
print(f"Participant responses : {len(df)}")

## 1.2 Screen the Pretest Sample

Consent, age, and the instructed-response attention check are converted to numeric form before exclusions are applied.

In [ ]:
for column in ["Consent ", "Age ", "Gender ", "D_Evaluation_6"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

screening = pd.DataFrame({
    "Criterion": [
        "Participant responses",
        "Did not provide consent",
        "Younger than 18 years",
        "Failed attention check among retained adults",
        "Final analytical sample",
    ],
    "n": [
        len(df),
        int((df["Consent "] != 1).sum()),
        int(((df["Consent "] == 1) & (df["Age "] < 18)).sum()),
        int(((df["Consent "] == 1) & (df["Age "] >= 18) & (df["D_Evaluation_6"] != 6)).sum()),
        int(((df["Consent "] == 1) & (df["Age "] >= 18) & (df["D_Evaluation_6"] == 6)).sum()),
    ],
})

display(screening)

df_analysis = df.loc[
    (df["Consent "] == 1)
    & (df["Age "] >= 18)
    & (df["D_Evaluation_6"] == 6)
].copy().reset_index(drop=True)

print(f"✓ Final analytical sample: N = {len(df_analysis)}")

## 1.3 Sample Description

Age and gender are summarized for the final analytical sample. Gender is coded according to the verified pretest questionnaire mapping (1 = Female, 2 = Male).


In [ ]:
GENDER_MAPPING = {1: "Female", 2: "Male"}

age_summary = pd.DataFrame({
    "N": [df_analysis["Age "].count()],
    "Mean": [df_analysis["Age "].mean()],
    "SD": [df_analysis["Age "].std(ddof=1)],
    "Min": [df_analysis["Age "].min()],
    "Max": [df_analysis["Age "].max()],
})

raw_gender_counts = (
    df_analysis["Gender "]
    .value_counts()
    .sort_index()
    .rename_axis("Raw Gender Code")
    .reset_index(name="n")
)
raw_gender_counts["%"] = raw_gender_counts["n"] / len(df_analysis) * 100

gender_summary = (
    df_analysis["Gender "]
    .map(GENDER_MAPPING)
    .value_counts()
    .rename_axis("Gender")
    .reset_index(name="n")
)
gender_summary["%"] = gender_summary["n"] / len(df_analysis) * 100

display(age_summary.round(2))
display(gender_summary)

print(
    f"N = {len(df_analysis)}; "
    f"M_age = {df_analysis['Age '].mean():.2f}; "
    f"SD_age = {df_analysis['Age '].std(ddof=1):.2f}"
)


# 2. Variable Construction

Each candidate product was evaluated with seven selection criteria. The resulting suitability score is treated as a **formative selection index**, not as a reflective psychometric scale: familiarity, liking, online purchase plausibility, current need, low deliberation, low brand dependence, and low prior ownership jointly define whether a product is suitable for the experiment rather than reflect one latent construct.

Accordingly, internal-consistency reliability such as Cronbach's alpha is not used as a criterion for this index. The item-level results in §3.2 provide the substantive basis for judging how each product performs across the individual selection criteria.

## 2.1 Product Item Mapping and Reverse Coding

The three product blocks are mapped according to the verified Qualtrics survey flow: `P_2` to wireless over-ear headphones, `P_3` to the insulated water bottle, and `P_1` to the Bluetooth speaker.

Items 5–7 are reverse-coded on the seven-point scale using `8 - response`. The reverse-coded values are stored in new columns with an `_R` suffix, ensuring that re-running the cell does not alter the original responses or double-reverse the items.


In [ ]:
PRODUCT_BLOCK_MAPPING = {
    "Wireless Over-Ear Headphones": [f"P_2 Evaluation _{i}" for i in range(1, 8)],
    "Insulated Water Bottle": [f"P_3 Evaluation_{i}" for i in range(1, 8)],
    "Bluetooth Speaker": [f"P_1 Evaluation _{i}" for i in range(1, 8)],
}

product_columns = PRODUCT_BLOCK_MAPPING

item_labels = {
    1: "Familiarity",
    2: "Liking",
    3: "Online Purchase Plausibility",
    4: "Current Need",
    5: "Low Deliberation (R)",
    6: "Low Brand Dependence (R)",
    7: "Low Prior Ownership (R)",
}

for product, columns in product_columns.items():
    df_analysis[columns] = df_analysis[columns].apply(pd.to_numeric, errors="coerce")

    scored_columns = []
    for i, column in enumerate(columns, start=1):
        if i <= 4:
            scored_columns.append(column)
        else:
            reverse_column = f"{column}_R"
            df_analysis[reverse_column] = 8 - df_analysis[column]
            scored_columns.append(reverse_column)

    score_name = product.replace(" ", "_").replace("-", "_") + "_Score"
    df_analysis[score_name] = df_analysis[scored_columns].mean(axis=1)

print("✓ Product items prepared; reverse-coded items stored in separate _R columns.")


## 2.2 Discount Variables

The discount-evaluation items are converted to numeric values. The instructed-response item is used only for screening and is not part of the substantive discount evaluation.

In [ ]:
discount_labels = {
    "D_Evaluation_1": "Attractiveness",
    "D_Evaluation_2": "Realism",
    "D_Evaluation_3": "Increase in Product Interest",
    "D_Evaluation_4": "Too Small to Matter",
    "D_Evaluation_5": "Strong Purchase-Decision Influence",
}

discount_columns = list(discount_labels)

df_analysis[discount_columns] = df_analysis[discount_columns].apply(
    pd.to_numeric, errors="coerce"
)

print("✓ Discount variables prepared.")

## 2.3 Chapter Summary

Product-suitability composites and the substantive discount-evaluation variables are prepared for the pretest analyses.

# 3. Pretest Analyses

This chapter reports the analyses used to select the focal product and evaluate the intended 15% discount.

## 3.1 Product Evaluation

Composite suitability scores are summarized for the three candidate products.

In [ ]:
score_columns = {
    product: product.replace(" ", "_").replace("-", "_") + "_Score"
    for product in product_columns
}

product_summary = pd.DataFrame([
    {
        "Product": product,
        "N": df_analysis[column].count(),
        "Mean": df_analysis[column].mean(),
        "SD": df_analysis[column].std(ddof=1),
    }
    for product, column in score_columns.items()
]).sort_values("Mean", ascending=False).reset_index(drop=True)

display(product_summary.round(2))

## 3.2 Item-Level Product Results

Item-level means and standard deviations provide the detailed product comparison reported in Appendix A.

In [ ]:
item_rows = []

for product, columns in product_columns.items():
    for i, column in enumerate(columns, start=1):
        scored_column = column if i <= 4 else f"{column}_R"
        item_rows.append({
            "Product": product,
            "Item": item_labels[i],
            "Mean": df_analysis[scored_column].mean(),
            "SD": df_analysis[scored_column].std(ddof=1),
        })

item_level_results = pd.DataFrame(item_rows)

display(
    item_level_results
    .pivot(index="Item", columns="Product", values="Mean")
    .round(2)
)

## 3.3 Repeated-Measures Product Comparison

A one-factor repeated-measures ANOVA compares the three product-suitability composites because every respondent evaluated all three candidate products.

In [ ]:
anova_long = pd.DataFrame({
    "Participant": np.repeat(np.arange(1, len(df_analysis) + 1), len(score_columns)),
    "Product": np.tile(list(score_columns.keys()), len(df_analysis)),
    "Suitability": np.column_stack(
        [df_analysis[column].to_numpy() for column in score_columns.values()]
    ).reshape(-1),
})

anova_model = AnovaRM(
    data=anova_long,
    depvar="Suitability",
    subject="Participant",
    within=["Product"],
).fit()

anova_table = anova_model.anova_table.reset_index().rename(columns={"index": "Effect"})

# Partial eta squared from F and its numerator/denominator degrees of freedom.
anova_table["Partial Eta Squared"] = (
    anova_table["F Value"] * anova_table["Num DF"]
    / (anova_table["F Value"] * anova_table["Num DF"] + anova_table["Den DF"])
)

display(anova_table.round(4))

row = anova_table.iloc[0]
print(
    f"Repeated-measures ANOVA: "
    f"F({row['Num DF']:.0f}, {row['Den DF']:.0f}) = {row['F Value']:.2f}, "
    f"p = {row['Pr > F']:.3f}, "
    f"partial η² = {row['Partial Eta Squared']:.2f}"
)

## 3.4 Discount Evaluation

The five substantive discount-evaluation items are summarized and tested against the seven-point scale midpoint of 4 using one-sample t-tests. These tests indicate whether respondents evaluated each aspect of the 15% discount as meaningfully above or below a neutral midpoint rather than relying on the descriptive means alone.

In [ ]:
discount_rows = []

for column, label in discount_labels.items():
    values = df_analysis[column].dropna()
    test = stats.ttest_1samp(values, popmean=4)

    discount_rows.append({
        "Measure": label,
        "N": values.count(),
        "Mean": values.mean(),
        "SD": values.std(ddof=1),
        "t": test.statistic,
        "df": len(values) - 1,
        "p": test.pvalue,
        "Direction vs. Midpoint": (
            "Above" if values.mean() > 4
            else "Below" if values.mean() < 4
            else "Equal"
        ),
        "Significant (p < .05)": "Yes" if test.pvalue < .05 else "No",
    })

discount_summary = pd.DataFrame(discount_rows)

display(discount_summary.round({
    "Mean": 2,
    "SD": 2,
    "t": 2,
    "p": 3,
}))


# 4. Export of Results

The main pretest tables and the cleaned analytical dataset are exported for documentation and manuscript use.

## 4.1 Export Tables and Analytical Data

The main pretest tables and the analytical dataset are exported for documentation and manuscript use. Original responses to the reverse-worded product items are retained under their source column names, while the scored versions are stored separately with an `_R` suffix. This makes the transformation explicit in the exported dataset.

In [ ]:
EXCEL_FILE = TABLE_DIR / "Pretest_Tables.xlsx"
ANALYSIS_FILE = OUTPUT_DIR / "Pretest_Analysis_Data.csv"

with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:
    screening.to_excel(writer, sheet_name="Screening", index=False)
    age_summary.to_excel(writer, sheet_name="Sample_Age", index=False)
    raw_gender_counts.to_excel(writer, sheet_name="Gender_Raw_Codes", index=False)
    gender_summary.to_excel(writer, sheet_name="Sample_Gender", index=False)
    product_summary.to_excel(writer, sheet_name="Product_Summary", index=False)
    item_level_results.to_excel(writer, sheet_name="Product_Item_Level", index=False)
    anova_table.to_excel(writer, sheet_name="Product_ANOVA", index=False)
    discount_summary.to_excel(writer, sheet_name="Discount_Summary", index=False)

df_analysis.to_csv(ANALYSIS_FILE, index=False)

print(f"✓ Tables exported to: {EXCEL_FILE}")
print(f"✓ Analytical dataset exported to: {ANALYSIS_FILE}")